# 3. Feature Engineering

## Objective
This notebook creates technical, derivatives, and statistical features
from the cleaned intraday NIFTY dataset.

Features include:
- EMA indicators for trading signals
- Options Greeks using Black–Scholes model
- Implied volatility metrics
- Put–Call ratios
- Futures basis and returns

Output:
- Final feature dataset for strategy and ML modeling

In [35]:
import pandas as pd
import numpy as np

In [36]:
df = pd.read_csv("../data/cleaned/nifty_merged_intraday.csv", low_memory=False)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp")

In [50]:
# ===============================
# FIX PE COLUMN MAPPING
# ===============================

# Price fields
df["open_pe"]  = df["open"]
df["high_pe"]  = df["high"]
df["low_pe"]   = df["low"]
df["close_pe"] = df["close"]

# Volume & OI
df["open_interest_pe"] = df["open_interest"]
df["contracts_pe"]     = df["contracts"]

# Safety: ensure numeric
pe_numeric_cols = [
    "open_pe", "high_pe", "low_pe", "close_pe",
    "open_interest_pe", "contracts_pe"
]

for col in pe_numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

### TASK 2.1 — EMA INDICATORS

In [51]:
df["ema_5"] = df["close_fut"].ewm(span=5, adjust=False).mean()
df["ema_15"] = df["close_fut"].ewm(span=15, adjust=False).mean()

In [52]:
df["expiry"] = pd.to_datetime(df["expiry"])
df["timestamp"] = pd.to_datetime(df["timestamp"])

df["time_to_expiry"] = (
    (df["expiry"] - df["timestamp"]).dt.total_seconds()
) / (365 * 24 * 60 * 60)

df = df[df["time_to_expiry"] > 0].copy()

### TASK 2.2 — OPTIONS GREEKS

In [53]:
from py_vollib.black_scholes.implied_volatility import implied_volatility
from py_vollib.black_scholes.greeks.analytical import delta, gamma, vega, theta, rho

In [54]:
def compute_iv_ce(row):
    try:
        return implied_volatility(
            price=row["close_ce"],
            S=row["close_fut"],
            K=row["strike"],
            r=0.065,
            t=row["time_to_expiry"],
            flag="c"
        )
    except:
        return np.nan

df["iv_ce"] = df.apply(compute_iv_ce, axis=1)

In [63]:
df["delta_ce"] = df.apply(
    lambda r: delta("c", r["close_fut"], r["strike"], r["time_to_expiry"], 0.065, r["iv_ce"])
    if pd.notna(r["iv_ce"]) else np.nan,
    axis=1
)

df["gamma_ce"] = df.apply(
    lambda r: gamma("c", r["close_fut"], r["strike"], r["time_to_expiry"], 0.065, r["iv_ce"])
    if pd.notna(r["iv_ce"]) else np.nan,
    axis=1
)

df["vega_ce"] = df.apply(
    lambda r: vega("c", r["close_fut"], r["strike"], r["time_to_expiry"], 0.065, r["iv_ce"])
    if pd.notna(r["iv_ce"]) else np.nan,
    axis=1
)
df["theta_ce"] = df.apply(
    lambda r: theta("c", r["close_fut"], r["strike"], r["time_to_expiry"], 0.065, r["iv_ce"])
    if pd.notna(r["iv_ce"]) else np.nan,
    axis=1
)
df["rho_ce"] = df.apply(
    lambda r: rho("c", r["close_fut"], r["strike"], r["time_to_expiry"], 0.065, r["iv_ce"])
    if pd.notna(r["iv_ce"]) else np.nan,
    axis=1
)

In [64]:
def compute_iv_pe(row):
    try:
        return implied_volatility(
            price=row["close_pe"],
            S=row["close_fut"],
            K=row["strike_pe"],
            r=0.065,
            t=row["time_to_expiry"],
            flag="p"
        )
    except:
        return np.nan

df["iv_pe"] = df.apply(compute_iv_pe, axis=1)

In [65]:
df["delta_pe"] = df.apply(
    lambda r: delta("p", r["close_fut"], r["strike_pe"], r["time_to_expiry"], 0.065, r["iv_pe"])
    if pd.notna(r["iv_pe"]) else np.nan,
    axis=1
)

df["gamma_pe"] = df.apply(
    lambda r: gamma("p", r["close_fut"], r["strike_pe"], r["time_to_expiry"], 0.065, r["iv_pe"])
    if pd.notna(r["iv_pe"]) else np.nan,
    axis=1
)

df["vega_pe"] = df.apply(
    lambda r: vega("p", r["close_fut"], r["strike_pe"], r["time_to_expiry"], 0.065, r["iv_pe"])
    if pd.notna(r["iv_pe"]) else np.nan,
    axis=1
)

df["theta_pe"] = df.apply(
    lambda r: theta("p", r["close_fut"], r["strike_pe"], r["time_to_expiry"], 0.065, r["iv_pe"])
    if pd.notna(r["iv_pe"]) else np.nan,
    axis=1
)

df["rho_pe"] = df.apply(
    lambda r: rho("p", r["close_fut"], r["strike_pe"], r["time_to_expiry"], 0.065, r["iv_pe"])
    if pd.notna(r["iv_pe"]) else np.nan,
    axis=1
)


### TASK 2.3 — DERIVED FEATURES

In [66]:
from py_vollib.black_scholes import black_scholes as bs

In [67]:
df["avg_iv"] = (df["iv_ce"] + df["iv_pe"]) / 2
df["iv_spread"] = df["iv_ce"] - df["iv_pe"]

df["pcr_oi"] = df["open_interest_pe"] / df["open_interest_ce"]
df["pcr_volume"] = df["contracts_pe"] / df["contracts_ce"]

df["futures_basis"] = (df["close_fut"] - df["close_pe"]) / df["close_pe"]

df["spot_returns"] = df["close_pe"].pct_change()
df["futures_returns"] = df["close_fut"].pct_change()

df["delta_neutral_ratio"] = abs(df["delta_ce"]) / abs(df["delta_pe"])

df["gamma_exposure"] = df["close_pe"] * (
    df["gamma_ce"] + df["gamma_pe"]
) * (df["open_interest_ce"] + df["open_interest_pe"])

### TASK 2.4 — FINAL FEATURE SET

All engineered features including EMA indicators, option Greeks, implied volatility metrics,
and derived features were consolidated into a final feature matrix.

The dataset is aligned at 5-minute frequency and is suitable for regime detection,
strategy development, and machine learning models.

**Output File:**  
`data/final/nifty_features_5min.csv`


In [68]:
df.to_csv("../data/final/nifty_features_5min.csv", index=False)

In [69]:
df[
    [
        "iv_ce","iv_pe",
        "delta_ce","delta_pe",
        "gamma_ce","gamma_pe",
        "vega_ce","vega_pe",
        "open_interest_ce","open_interest_pe",
        "contracts_ce","contracts_pe"
    ]
].notna().sum()


iv_ce               225575
iv_pe               289973
delta_ce            225575
delta_pe            289973
gamma_ce            225575
gamma_pe            289973
vega_ce             225575
vega_pe             289973
open_interest_ce    289973
open_interest_pe    289973
contracts_ce        282362
contracts_pe        279678
dtype: int64

In [26]:
sorted(df.columns.tolist())

['atm_strike',
 'atm_strike_ce',
 'atm_strike_fut',
 'change_in_oi',
 'change_in_oi_ce',
 'change_in_oi_fut',
 'close',
 'close_ce',
 'close_fut',
 'contracts',
 'contracts_ce',
 'contracts_fut',
 'date',
 'date_ce',
 'date_fut',
 'delta_ce',
 'delta_pe',
 'ema_15',
 'ema_5',
 'expiry',
 'expiry_ce',
 'expiry_fut',
 'gamma_ce',
 'gamma_pe',
 'high',
 'high_ce',
 'high_fut',
 'iv_ce',
 'iv_pe',
 'low',
 'low_ce',
 'low_fut',
 'ltp',
 'ltp_ce',
 'ltp_fut',
 'open',
 'open_ce',
 'open_fut',
 'open_interest',
 'open_interest_ce',
 'open_interest_fut',
 'option_type',
 'option_type_pe',
 'premium_turnover__in___rs_lakhs',
 'premium_turnover__in___rs_lakhs_pe',
 'rho_ce',
 'rho_pe',
 'settle_price',
 'settle_price_ce',
 'settle_price_fut',
 'strike',
 'strike_pe',
 'symbol',
 'symbol_ce',
 'symbol_fut',
 'theta_ce',
 'theta_pe',
 'timestamp',
 'turnover__in___rs_lakhs',
 'turnover__in__rs_lakhs',
 'turnover__in__rs_lakhs_pe',
 'underlying_value',
 'underlying_value_ce',
 'underlying_value_fu